In [1]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "klue/bert-base"

c:\Users\shyuj\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  

c:\Users\shyuj\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shyuj\.cache\huggingface\hub\models--klue--bert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [3]:
sample_text = "이 영화 정말 재미있다"

encoded = tokenizer(sample_text)

print(encoded)

{'input_ids': [2, 1504, 3771, 3944, 6001, 2062, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


In [4]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8389.37it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

In [5]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [6]:
!pip install sentence-transformers faiss-cpu


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [8]:
knowledge_base = [
    "이 영화는 2023년 개봉한 한국 액션 블록버스터로 관객 800만 명을 돌파했다.",
    "주연 배우의 연기력이 뛰어나다는 평이 많으며 CG 퀄리티가 훌륭하다.",
    "스토리가 진부하고 예측 가능하다는 부정적인 의견도 존재한다.",
    "OST가 매우 감동적이며 엔딩 크레딧까지 자리를 뜨지 못하게 만든다.",
    "러닝타임이 2시간 30분으로 다소 길지만 지루하지 않다는 반응이다.",
    "아이맥스 상영관에서 보면 몰입감이 극대화된다는 관람객 후기가 많다."
]

In [9]:
embedder = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2'
)

kb_embeddings = embedder.encode(
    knowledge_base,
    convert_to_numpy=True
)

c:\Users\shyuj\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shyuj\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00

In [10]:
dimension = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(kb_embeddings)

In [11]:
def retrieve(query, top_k=2):
    query_vec = embedder.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_vec,
        top_k
    )

    return [knowledge_base[i] for i in indices[0]]

In [15]:
from transformers import pipeline

fine_tuned_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

def format_result(result):
  
    if result['label'] == 'LABEL_0':
        return f"부정적 (확률: {result['score']:.2f})"
    elif result['label'] == 'LABEL_1':
        return f"긍정적 (확률: {result['score']:.2f})"
    else:
        return f"{result['label']} (확률: {result['score']:.2f})"

def rag_sentiment_pipeline(query):
    print(f"[질문]: {query}")

    retrieved = retrieve(query, top_k=2)
    context = " ".join(retrieved)
    print(f"[검색된 문서]: {context}")

    augmented_input = f"{context} {query}"
    result = fine_tuned_pipeline(augmented_input)[0]

    print(f"[감정 예측]: {format_result(result)}\n")
    print("-" * 60)

In [16]:
queries = ["이 영화 배우 연기가 어때요?","스토리는 어떤가요?","아이맥스로 볼 만한가요?"]

for q in queries:
    rag_sentiment_pipeline(q)

[질문]: 이 영화 배우 연기가 어때요?
[검색된 문서]: 아이맥스 상영관에서 보면 몰입감이 극대화된다는 관람객 후기가 많다. 주연 배우의 연기력이 뛰어나다는 평이 많으며 CG 퀄리티가 훌륭하다.
[감정 예측]: 부정적 (확률: 0.51)

------------------------------------------------------------
[질문]: 스토리는 어떤가요?
[검색된 문서]: 아이맥스 상영관에서 보면 몰입감이 극대화된다는 관람객 후기가 많다. 스토리가 진부하고 예측 가능하다는 부정적인 의견도 존재한다.
[감정 예측]: 부정적 (확률: 0.55)

------------------------------------------------------------
[질문]: 아이맥스로 볼 만한가요?
[검색된 문서]: 아이맥스 상영관에서 보면 몰입감이 극대화된다는 관람객 후기가 많다. OST가 매우 감동적이며 엔딩 크레딧까지 자리를 뜨지 못하게 만든다.
[감정 예측]: 부정적 (확률: 0.53)

------------------------------------------------------------
